# 03 - Precipitação: CHIRPS diário

| | |
|---|---|
| **Tarefa do plano** | S1-06 - CHIRPS 2006-2026 recortado à área de estudo (série diária) |
| **Variáveis (Parte I)** | #23 precipitação diária (base para #24-#26, derivadas na semana 2) |
| **Fonte** | CHIRPS (CHC/UCSB) via Earth Engine `UCSB-CHG/CHIRPS/DAILY` |
| **Saída** | `data/raw/precipitation/chirps/` |

Descarregado na grelha nativa de 0,05 graus sobre toda a bacia (AOI climática), um NetCDF por ano.

Correr primeiro `00_setup_and_authentication` e `01_study_area_aoi`. Todos os
downloads podem ser retomados: os ficheiros que já existem são saltados, e cada
ficheiro fica registado em `data/metadata/download_manifest.csv`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import ee
import pandas as pd
import matplotlib.pyplot as plt

from ete import config, datasets, gee, series
from ete import grid as G

cfg = config.load_config()
gee.init(cfg)

aoi = G.load_aoi(cfg)                       # AOI das imagens (EPSG:4326)
region = gee.ee_geometry(aoi)
grid30 = G.load_grid(cfg)                   # grelha do projeto a 30 m (EPSG:32736)
grid10 = G.load_grid(cfg, 10)               # grelha a 10 m, encaixada na de 30 m
climate_bounds = tuple(G.load_aoi(cfg, "aoi_climate").total_bounds)   # O, S, E, N
MASK = gee.ee_geometry(G.imagery_mask(cfg))  # distrito + margem: fora daqui as imagens mensais ficam sem dados

## 1. Download

In [ ]:
ds = cfg["datasets"]["chirps"]
OUT_DIR = config.path(cfg, "chirps")
grid = gee.collection_grid(ee.ImageCollection(ds["id"]), ds["band"], climate_bounds, fallback_res=0.05)
print(grid)

log = gee.download_daily_netcdf(cfg, "chirps", ds["id"], ds["band"], grid, OUT_DIR,
                                start=ds["start"], freq="YS", units="mm/day")
log

## 2. Controlo de qualidade e série da bacia

In [ ]:
rain = series.open_series(cfg, "chirps", ds["band"])
print("período:", str(rain.time.values[0])[:10], "a", str(rain.time.values[-1])[:10], "|", rain.sizes)
print("dias em falta:", series.missing_days(rain))
print("intervalo (mm/dia):", float(rain.min()), "a", float(rain.max()))

daily = pd.DataFrame({
    "basin_mm": series.area_mean(rain, G.load_aoi(cfg, "basin")),
    "district_mm": series.area_mean(rain, G.load_aoi(cfg, "district")),
})
csv = config.path(cfg, "interim", "precipitation", "chirps_daily_area_mean.csv")
daily.to_csv(csv, index_label="date")
print("escrito", csv.relative_to(ROOT))

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
daily.resample("YS").sum().set_axis(daily.resample("YS").sum().index.year).plot.bar(ax=axes[0])
axes[0].set_ylabel("mm / ano"); axes[0].set_title("Precipitação anual (o último ano pode estar incompleto)")
daily["basin_mm"].plot(ax=axes[1], linewidth=0.5)
axes[1].set_ylabel("mm / dia"); axes[1].set_title("Precipitação diária média da bacia")
plt.tight_layout()